In [ ]:
!!pip install faiss-cpu
"""
Notebook educativo sobre Retrieval-Augmented Generation (RAG)
Seminario Bourbaki — NLP Avanzado

Estructura:
  PARTE 1 — RAG con documentos sintéticos (revisión del notebook anterior)
  PARTE 2 — RAG con tus propios PDFs (pipeline completo de producción)
"""

# ==============================================================================
# ⚙️ PARÁMETROS GLOBALES
# Modifica aquí para controlar todo el comportamiento del notebook
# ==============================================================================

# --- Modelo de embeddings ---
EMBEDDING_MODEL    = "text-embedding-3-small"   # 1536 dims, económico y preciso
EMBEDDING_DIM      = 1536                        # Debe coincidir con el modelo

# --- Modelo de generación ---
GENERATION_MODEL   = "gpt-5"              # Cambiar a "gpt-4o" para mayor calidad

# --- RAG retrieval ---
TOP_K              = 3      # Cuántos fragmentos recuperar por query (1–10 típico)
CHUNK_SIZE         = 500    # Tamaño de cada fragmento en tokens (aprox.)
CHUNK_OVERLAP      = 50     # Solapamiento entre fragmentos consecutivos

# --- Rutas de datos ---
DATA_FOLDER        = "/content/data"            # Carpeta con tus PDFs en Colab
SUPPORTED_EXT      = ".pdf"

# --- Prompt del sistema ---
SYSTEM_PROMPT = (
    "Eres un asistente experto. Responde únicamente basándote en el contexto "
    "proporcionado. Si la respuesta no está en el contexto, indícalo claramente. "
    "Sé conciso y preciso."
)


# ==============================================================================
# 📦 INSTALACIÓN DE DEPENDENCIAS
# ==============================================================================

# !pip install faiss-cpu openai tiktoken pymupdf numpy scikit-learn --quiet

"""
Librerías utilizadas:
  - openai      : cliente para embeddings (text-embedding-3-small) y chat (gpt-4o-mini)
  - faiss-cpu   : índice vectorial de Meta para búsqueda eficiente por similitud
  - tiktoken    : tokenizador de OpenAI, para chunking preciso por tokens (no chars)
  - pymupdf     : extracción de texto de PDFs (más robusto que pdfminer o PyPDF2)
  - numpy       : operaciones vectoriales
  - scikit-learn: cosine_similarity para la demo del Nivel 2
"""


# ==============================================================================
# 🔑 AUTENTICACIÓN
# ==============================================================================

import getpass, os
from openai import OpenAI

api_key = getpass.getpass("🔑 Introduce tu OpenAI API Key: ")
client  = OpenAI(api_key=api_key)


# ==============================================================================
# ═══════════════════════════════════════════════════════════════════════════════
#  PARTE 1 — RAG CON DOCUMENTOS SINTÉTICOS (revisión del notebook anterior)
# ═══════════════════════════════════════════════════════════════════════════════
# ==============================================================================

"""
# ¿Qué es RAG?

RAG (Retrieval-Augmented Generation) es una arquitectura que combina dos sistemas:

  1. RETRIEVAL (Recuperación): dado un texto de consulta, busca los fragmentos
     más relevantes dentro de una base de conocimiento vectorizada.

  2. GENERATION (Generación): usa un LLM para generar una respuesta, pero
     *aumentando* su prompt con esos fragmentos recuperados como contexto.

El flujo completo es:

  Query del usuario
       │
       ▼
  [Embedding de la query]         ← text-embedding-3-small
       │
       ▼
  [Búsqueda en índice vectorial]  ← FAISS (similitud coseno / L2)
       │
       ▼
  [Top-K fragmentos relevantes]
       │
       ▼
  [Prompt = sistema + contexto + query]
       │
       ▼
  [LLM genera respuesta]          ← gpt-4o-mini
       │
       ▼
  Respuesta fundamentada

¿Por qué no simplemente pasarle todo al LLM?
  - Los context windows tienen límites (y costo).
  - El LLM "se distrae" con información irrelevante.
  - RAG escala a millones de documentos; el contexto full no.
"""


# ------------------------------------------------------------------------------
# NIVEL 1 — Naive RAG: todo el contexto en el prompt
# ------------------------------------------------------------------------------

"""
### Nivel 1 — Naive RAG

El enfoque más simple: concatenar todos los documentos y pasarlos directamente
al LLM. Funciona bien para colecciones pequeñas (< ~20 documentos cortos),
pero no escala.
"""

documentos = [
    "La política de GlobalCorp permite 2 días de teletrabajo a la semana.",
    "Las solicitudes se hacen en el portal WorkLife con 48h de antelación.",
    "El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.",
    "El contacto oficial para dudas es rrhh@globalcorp.com."
]

pregunta_1 = "¿Cuántos días puedo trabajar desde casa?"
contexto_completo = "\n".join(documentos)

respuesta_naive = client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Contexto:\n{contexto_completo}\n\nPregunta: {pregunta_1}"}
    ]
)

print("=" * 60)
print("NIVEL 1 — Naive RAG")
print(f"Pregunta: {pregunta_1}")
print(f"Respuesta: {respuesta_naive.choices[0].message.content}")
print(f"Tokens usados: {respuesta_naive.usage.total_tokens}")


# ------------------------------------------------------------------------------
# NIVEL 2 — Semantic Retrieval: embeddings + similitud coseno
# ------------------------------------------------------------------------------

"""
### Nivel 2 — Semantic Retrieval

En lugar de pasar todo el contexto, convertimos cada documento a un vector
(embedding) y calculamos qué tan "cercana" es la pregunta a cada documento.

¿Qué es un embedding?
  Un embedding es una representación numérica densa del significado de un texto.
  text-embedding-3-small convierte cualquier texto a un vector de 1536 números.
  Textos semánticamente similares tienen vectores geométricamente cercanos.

Similitud coseno:
  Mide el ángulo entre dos vectores (ignora la magnitud).
  Valor de 1.0 = idénticos, 0.0 = ortogonales (sin relación).
"""

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_embedding(text: str) -> list[float]:
    """Convierte texto a vector usando el modelo configurado."""
    response = client.embeddings.create(
        input=[text],
        model=EMBEDDING_MODEL
    )
    return response.data[0].embedding

# Vectorizar los documentos (en producción esto se hace UNA vez y se persiste)
print(f"\nVectorizando {len(documentos)} documentos con {EMBEDDING_MODEL}...")
embeddings_docs = [get_embedding(doc) for doc in documentos]

# Vectorizar la pregunta
pregunta_2       = "¿A quién contacto si tengo problemas?"
embedding_pregunta = get_embedding(pregunta_2)

# Calcular similitudes y seleccionar el mejor
scores  = cosine_similarity([embedding_pregunta], embeddings_docs)[0]
top_idx = np.argmax(scores)

print("\n" + "=" * 60)
print("NIVEL 2 — Semantic Retrieval")
print(f"Pregunta: {pregunta_2}")
for i, (doc, score) in enumerate(zip(documentos, scores)):
    marker = " ◄ SELECCIONADO" if i == top_idx else ""
    print(f"  [{score:.4f}] {doc[:60]}...{marker}" if len(doc) > 60 else f"  [{score:.4f}] {doc}{marker}")


# ------------------------------------------------------------------------------
# NIVEL 3 — FAISS: índice vectorial eficiente + pipeline RAG completo
# ------------------------------------------------------------------------------

"""
### Nivel 3 — FAISS + RAG Completo

FAISS (Facebook AI Similarity Search) es una librería de Meta que indexa
vectores para búsquedas de vecinos más cercanos de forma muy eficiente.

¿Por qué FAISS en lugar de sklearn?
  - sklearn.cosine_similarity es O(n): compara la query contra TODOS los vectores.
  - FAISS con índices como IVF o HNSW es sub-lineal: agrupa vectores en clusters
    y solo busca dentro de los clusters relevantes.
  - Con millones de documentos la diferencia es de segundos vs horas.

Índice que usamos: IndexFlatL2
  - Exacto (no aproximado): revisa todos los vectores.
  - Usa distancia euclidiana (L2).
  - Equivalente a coseno cuando los vectores están normalizados (como los de OpenAI).
  - Para escala mayor: IndexIVFFlat (clusters) o IndexHNSW (grafo jerárquico).

El pipeline completo R-A-G:
  R = Retrieval  : FAISS encuentra los top-K fragmentos más cercanos.
  A = Augmentation: se construye un prompt enriquecido con esos fragmentos.
  G = Generation : el LLM genera la respuesta fundamentada en el contexto.
"""

import faiss

# Construir el índice FAISS
index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(np.array(embeddings_docs, dtype='float32'))
print(f"\nÍndice FAISS creado. Vectores indexados: {index.ntotal}")

def rag_query(pregunta: str, k: int = TOP_K, verbose: bool = True) -> str:
    """
    Pipeline RAG completo:
      1. Embed la pregunta.
      2. Busca los k fragmentos más cercanos en FAISS.
      3. Construye un prompt aumentado.
      4. Genera y retorna la respuesta.
    """
    # R — Retrieval
    query_vec = np.array([get_embedding(pregunta)], dtype='float32')
    D, I      = index.search(query_vec, k=min(k, index.ntotal))

    fragmentos_recuperados = [documentos[i] for i in I[0]]
    contexto_rag = "\n".join(f"[{j+1}] {frag}" for j, frag in enumerate(fragmentos_recuperados))

    # A — Augmentation
    prompt_usuario = f"Contexto:\n{contexto_rag}\n\nPregunta: {pregunta}"

    # G — Generation
    respuesta = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt_usuario}
        ]
    )

    if verbose:
        print("\n" + "=" * 60)
        print(f"Pregunta: {pregunta}")
        print(f"\nFragmentos recuperados (top-{k}):")
        for j, frag in enumerate(fragmentos_recuperados):
            print(f"  [{j+1}] {frag}")
        print(f"\nRespuesta ({GENERATION_MODEL}):")
        print(f"  {respuesta.choices[0].message.content}")

    return respuesta.choices[0].message.content

# Ejemplo
rag_query("Necesito teletrabajo pero llevo solo 1 mes en la empresa.")
rag_query("Como puedo aprender a programar en Python")




🔑 Introduce tu OpenAI API Key: ··········
NIVEL 1 — Naive RAG
Pregunta: ¿Cuántos días puedo trabajar desde casa?
Respuesta: Hasta 2 días a la semana, siempre que no estés en los 3 primeros meses de periodo de prueba. Debes solicitarlo en WorkLife con 48 horas de antelación. Para dudas: rrhh@globalcorp.com.
Tokens usados: 444

Vectorizando 4 documentos con text-embedding-3-small...

NIVEL 2 — Semantic Retrieval
Pregunta: ¿A quién contacto si tengo problemas?
  [0.2071] La política de GlobalCorp permite 2 días de teletrabajo a la...
  [0.3075] Las solicitudes se hacen en el portal WorkLife con 48h de an...
  [0.1529] El periodo de prueba de 3 meses excluye el beneficio de tele...
  [0.4789] El contacto oficial para dudas es rrhh@globalcorp.com. ◄ SELECCIONADO

Índice FAISS creado. Vectores indexados: 4

Pregunta: Necesito teletrabajo pero llevo solo 1 mes en la empresa.

Fragmentos recuperados (top-3):
  [1] El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.
  [2] La po

'El contexto no incluye información sobre cómo aprender a programar en Python, así que no puedo responder con los datos disponibles.  \nPara consultas oficiales relacionadas con la empresa, el contacto es rrhh@globalcorp.com.'

In [ ]:
# -*- coding: utf-8 -*-
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          RAG sobre el Diario Oficial de la Federación (DOF)                 ║
║          Seminario Bourbaki — NLP Avanzado                                  ║
╚══════════════════════════════════════════════════════════════════════════════╝

Pipeline completo de Retrieval-Augmented Generation aplicado a artículos
reales del DOF. El notebook está dividido en tres bloques independientes:

  BLOQUE 1 — Descarga de artículos del DOF como PDFs
  BLOQUE 2 — Construcción del índice vectorial (embeddings + FAISS)
  BLOQUE 3 — Consultas RAG: preguntas de ejemplo + modo interactivo
"""


# ──────────────────────────────────────────────────────────────────────────────
# 📦  INSTALACIÓN DE DEPENDENCIAS
# ──────────────────────────────────────────────────────────────────────────────
# Ejecutar una sola vez al inicio del entorno de Colab.
#
#  pymupdf     → extracción de texto de PDFs (fitz)
#  tiktoken    → tokenizador oficial de OpenAI, para chunking preciso
#  faiss-cpu   → índice vectorial de Meta para búsqueda por similitud
#  reportlab   → generación de PDFs con soporte UTF-8 nativo
#  openai      → cliente para embeddings y chat completions
#  requests    → descarga de páginas web del DOF
#  beautifulsoup4 → parseo del HTML del DOF

# !pip install pymupdf tiktoken faiss-cpu reportlab openai requests beautifulsoup4 --quiet


# ──────────────────────────────────────────────────────────────────────────────
# ⚙️  PARÁMETROS GLOBALES
# ──────────────────────────────────────────────────────────────────────────────
# Todos los valores configurables están aquí arriba. Modificar este bloque
# es suficiente para adaptar el pipeline a otros documentos o modelos.

# Modelo de embeddings — convierte texto a vectores numéricos.
# text-embedding-3-small produce vectores de 1536 dimensiones.
# Es el modelo más económico de OpenAI con buena precisión semántica.
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM   = 1536   # debe coincidir con el modelo elegido

# Modelo de generación — recibe el contexto recuperado y produce la respuesta.
# gpt-4o-mini es rápido y económico; cambiar a "gpt-4o" para mayor calidad.
GENERATION_MODEL = "gpt-4o-mini"

# Número de fragmentos a recuperar por consulta.
# Valores típicos en producción: entre 3 y 10.
# Más fragmentos = más contexto, pero también más tokens y costo.
TOP_K = 3

# Tamaño de cada fragmento en tokens al dividir las páginas.
# 500 tokens ≈ ~375 palabras en español. Un valor más pequeño mejora
# la precisión del retrieval; uno más grande preserva más contexto local.
CHUNK_SIZE = 500

# Solapamiento entre fragmentos consecutivos.
# Evita perder información en los cortes (ej. una oración que cruza el límite).
CHUNK_OVERLAP = 50

# Carpeta donde se guardan y leen los PDFs en Colab.
DATA_FOLDER   = "/content/data"
SUPPORTED_EXT = ".pdf"

# Instrucción de sistema para el LLM.
# Le indica que solo responda con base en el contexto recuperado.
SYSTEM_PROMPT = (
    "Eres un asistente experto en normativa y regulación mexicana. "
    "Responde únicamente basándote en el contexto proporcionado del "
    "Diario Oficial de la Federación. Cita la fuente y página cuando sea posible. "
    "Si la respuesta no está en el contexto, indícalo claramente."
)


# ──────────────────────────────────────────────────────────────────────────────
# 🔑  AUTENTICACIÓN
# ──────────────────────────────────────────────────────────────────────────────

import getpass, os
from openai import OpenAI

api_key = getpass.getpass("🔑 Introduce tu OpenAI API Key: ")
client  = OpenAI(api_key=api_key)


# ══════════════════════════════════════════════════════════════════════════════
#  BLOQUE 1 — DESCARGA DE ARTÍCULOS DEL DOF
# ══════════════════════════════════════════════════════════════════════════════
"""
El DOF expone cada artículo en una URL pública con la estructura:
  https://www.dof.gob.mx/nota_detalle_popup.php?codigo={CODIGO}

El código lo encuentras abriendo cualquier artículo en dof.gob.mx
y copiando el número en ?codigo=XXXXXXX de la URL.

El HTML contiene el texto dentro de una tabla principal (<table>),
que parseamos con BeautifulSoup y luego convertimos a PDF con reportlab.

Nota sobre verify=False:
  El servidor del DOF tiene un certificado SSL con cadena incompleta
  (error del lado del gobierno, no nuestro). Deshabilitamos la verificación
  porque solo leemos contenido público — no transmitimos datos sensibles.
"""

import requests, time, urllib3
from bs4 import BeautifulSoup
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.enums import TA_CENTER

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Lista de artículos a descargar.
# Agrega o quita entradas según los documentos que necesites.
ARTICULOS_DOF = [
    {"codigo": "5780977", "nombre": "SHCP_aguas_nacionales_feb2026"},
    {"codigo": "5779201", "nombre": "CONDUSEF_dias_inhabiles_ene2026"},
    {"codigo": "5778252", "nombre": "DOF_OAJ_titulares_ene2026"},
    {"codigo": "5660409", "nombre": "Comision_Mares_Costas_ago2022"},
    {"codigo": "5272787", "nombre": "NOM004_expediente_clinico"},
]

BASE_URL_DOF = "https://www.dof.gob.mx/nota_detalle_popup.php?codigo={}"
HEADERS_DOF  = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}
SLEEP_SEC    = 2   # pausa entre requests para no saturar el servidor del DOF

os.makedirs(DATA_FOLDER, exist_ok=True)


def scrape_articulo_dof(codigo: str) -> dict:
    """
    Descarga el HTML de un artículo del DOF y extrae su texto.

    El contenido real está dentro de un <table> en el popup.
    Usamos encoding utf-8 explícito porque el header del DOF declara
    iso-8859-1 pero el contenido puede incluir caracteres UTF-8.
    """
    url      = BASE_URL_DOF.format(codigo)
    response = requests.get(url, headers=HEADERS_DOF, timeout=15, verify=False)
    response.raise_for_status()
    response.encoding = "utf-8"

    soup      = BeautifulSoup(response.text, "html.parser")
    titulo    = soup.find("title")
    titulo    = titulo.get_text(strip=True) if titulo else f"DOF_{codigo}"
    contenido = soup.find("table")
    texto     = (contenido.get_text(separator="\n", strip=True)
                 if contenido else soup.body.get_text(separator="\n", strip=True))

    return {"codigo": codigo, "titulo": titulo, "texto": texto, "url": url}


def texto_a_pdf_dof(articulo: dict, ruta_salida: str) -> None:
    """
    Convierte el texto extraído a un PDF usando reportlab.

    Usamos reportlab (no fpdf2) porque maneja UTF-8 nativo:
    acentos, ñ y otros caracteres del español no requieren ningún
    workaround de encoding.

    Los caracteres &, <, > se escapan porque reportlab usa un parser
    XML interno para renderizar párrafos.
    """
    doc = SimpleDocTemplate(
        ruta_salida, pagesize=A4,
        leftMargin=15*mm, rightMargin=15*mm,
        topMargin=15*mm,  bottomMargin=15*mm
    )

    estilo_header = ParagraphStyle("header", fontSize=11, fontName="Helvetica-Bold",
                                   alignment=TA_CENTER, spaceAfter=4)
    estilo_meta   = ParagraphStyle("meta",   fontSize=8,  fontName="Helvetica",
                                   alignment=TA_CENTER, spaceAfter=8)
    estilo_titulo = ParagraphStyle("titulo", fontSize=10, fontName="Helvetica-Bold",
                                   spaceAfter=6)
    estilo_cuerpo = ParagraphStyle("cuerpo", fontSize=9,  fontName="Helvetica",
                                   leading=13, spaceAfter=2)

    elementos = [
        Paragraph("Diario Oficial de la Federación", estilo_header),
        Paragraph(f"Código: {articulo['codigo']}  |  {articulo['url']}", estilo_meta),
        Paragraph(articulo["titulo"], estilo_titulo),
        Spacer(1, 4*mm),
    ]

    for linea in articulo["texto"].split("\n"):
        linea = linea.strip()
        if not linea:
            continue
        linea = linea.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        elementos.append(Paragraph(linea, estilo_cuerpo))

    doc.build(elementos)


# --- Ejecutar descarga ---
print(f"📥 Descargando {len(ARTICULOS_DOF)} artículos del DOF → {DATA_FOLDER}\n")

for art in ARTICULOS_DOF:
    try:
        print(f"  → {art['nombre']} (código {art['codigo']})...", end=" ")
        datos    = scrape_articulo_dof(art["codigo"])
        ruta_pdf = os.path.join(DATA_FOLDER, f"{art['nombre']}.pdf")
        texto_a_pdf_dof(datos, ruta_pdf)
        print(f"✓  ({os.path.getsize(ruta_pdf)/1024:.1f} KB)")
        time.sleep(SLEEP_SEC)
    except Exception as e:
        print(f"✗  Error: {e}")

print(f"\n✅ PDFs listos en {DATA_FOLDER}/")


# ══════════════════════════════════════════════════════════════════════════════
#  BLOQUE 2 — CONSTRUCCIÓN DEL ÍNDICE VECTORIAL
# ══════════════════════════════════════════════════════════════════════════════
"""
Este bloque transforma los PDFs en una estructura de búsqueda semántica.
Los pasos son:

  1. Extracción — PyMuPDF lee cada página del PDF y devuelve texto plano.
                  Guardamos la fuente y número de página como metadata.

  2. Chunking   — Cada página se divide en fragmentos de CHUNK_SIZE tokens
                  con CHUNK_OVERLAP tokens de solapamiento.
                  Usamos tiktoken (tokenizador de OpenAI) para contar tokens
                  exactos, no caracteres. 1 token ≈ 0.75 palabras en español.

  3. Embeddings — Cada fragmento se convierte a un vector de 1536 números
                  con text-embedding-3-small. Textos semánticamente similares
                  producen vectores geométricamente cercanos.

  4. FAISS      — Los vectores se indexan con IndexFlatL2 de FAISS.
                  Este índice permite encontrar los K vectores más cercanos
                  a una query en microsegundos, sin revisar todos los vectores
                  uno por uno como haría sklearn.
"""

import numpy as np
import pymupdf as fitz
import tiktoken
import faiss
from pathlib import Path


def get_embedding(text: str) -> list[float]:
    """Convierte un texto a vector numérico usando EMBEDDING_MODEL."""
    return client.embeddings.create(input=[text], model=EMBEDDING_MODEL).data[0].embedding


def extraer_texto_pdf(ruta_pdf: str) -> list[dict]:
    """
    Extrae el texto de un PDF página por página con PyMuPDF.
    Retorna lista de dicts: { fuente, pagina, texto }
    """
    nombre, paginas = Path(ruta_pdf).name, []
    with fitz.open(ruta_pdf) as doc:
        for num_pag, pagina in enumerate(doc, start=1):
            texto = pagina.get_text().strip()
            if texto:   # ignorar páginas en blanco
                paginas.append({"fuente": nombre, "pagina": num_pag, "texto": texto})
    print(f"  ✓ {nombre}: {len(paginas)} páginas con texto")
    return paginas


def cargar_pdfs(carpeta: str) -> list[dict]:
    """Carga todos los PDFs de la carpeta y retorna lista de páginas."""
    archivos = sorted(Path(carpeta).glob(f"*{SUPPORTED_EXT}"))
    if not archivos:
        raise FileNotFoundError(
            f"No se encontraron PDFs en {carpeta}. "
            "Asegúrate de haber ejecutado el Bloque 1 primero."
        )
    print(f"\n📂 {len(archivos)} PDFs encontrados en {carpeta}:")
    todas = []
    for archivo in archivos:
        todas.extend(extraer_texto_pdf(str(archivo)))
    print(f"\nTotal: {len(todas)} páginas extraídas")
    return todas


tokenizer = tiktoken.get_encoding("cl100k_base")  # encoding de los modelos OpenAI


def chunk_por_tokens(texto: str) -> list[str]:
    """
    Divide un texto en fragmentos de CHUNK_SIZE tokens con CHUNK_OVERLAP
    tokens de solapamiento entre fragmentos consecutivos.
    """
    tokens, chunks, i = tokenizer.encode(texto), [], 0
    while i < len(tokens):
        chunks.append(tokenizer.decode(tokens[i : i + CHUNK_SIZE]))
        i += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def crear_fragmentos(paginas: list[dict]) -> list[dict]:
    """
    Aplica chunking a cada página y retorna lista plana de fragmentos.
    Cada fragmento conserva su metadata: fuente, página y chunk_idx.
    """
    fragmentos = []
    for pag in paginas:
        for idx, chunk in enumerate(chunk_por_tokens(pag["texto"])):
            fragmentos.append({
                "fuente"   : pag["fuente"],
                "pagina"   : pag["pagina"],
                "chunk_idx": idx,
                "texto"    : chunk
            })
    print(f"\n✂️  Chunking completado:")
    print(f"   Páginas     : {len(paginas)}")
    print(f"   Fragmentos  : {len(fragmentos)}")
    print(f"   Chunk size  : {CHUNK_SIZE} tokens  |  Overlap: {CHUNK_OVERLAP} tokens")
    return fragmentos


def vectorizar_fragmentos(fragmentos: list[dict], batch_size: int = 100) -> np.ndarray:
    """
    Convierte los fragmentos a embeddings en lotes para eficiencia.
    Retorna array numpy de shape (N, EMBEDDING_DIM).

    batch_size=100 es seguro y eficiente. El límite de la API de OpenAI
    es 2048 inputs por llamada; subir el batch reduce el número de requests.
    """
    textos, todos_emb = [f["texto"] for f in fragmentos], []
    print(f"\n🔢 Vectorizando {len(textos)} fragmentos con {EMBEDDING_MODEL}...")
    for i in range(0, len(textos), batch_size):
        resp = client.embeddings.create(input=textos[i:i+batch_size], model=EMBEDDING_MODEL)
        todos_emb.extend([item.embedding for item in resp.data])
        print(f"   {min(i+batch_size, len(textos))}/{len(textos)} vectorizados...", end="\r")
    print(f"\n   ✓ Array final: ({len(todos_emb)}, {len(todos_emb[0])})")
    return np.array(todos_emb, dtype='float32')


# --- Ejecutar pipeline de indexado ---

paginas    = cargar_pdfs(DATA_FOLDER)
fragmentos = crear_fragmentos(paginas)

embeddings_pdf = vectorizar_fragmentos(fragmentos)

# IndexFlatL2: índice exacto por distancia euclidiana (L2).
# Equivalente a similitud coseno cuando los vectores están normalizados,
# como los que produce text-embedding-3-small.
index_pdf = faiss.IndexFlatL2(EMBEDDING_DIM)
index_pdf.add(embeddings_pdf)

print(f"\n📦 Índice FAISS construido.")
print(f"   Vectores indexados : {index_pdf.ntotal}")
print(f"   Dimensión          : {EMBEDDING_DIM}")
print(f"   Tipo de índice     : IndexFlatL2 (búsqueda exacta)")


# ══════════════════════════════════════════════════════════════════════════════
#  BLOQUE 3 — CONSULTAS RAG
# ══════════════════════════════════════════════════════════════════════════════
"""
Con el índice construido, el pipeline RAG para cada consulta es:

  R — Retrieval:
      La query se convierte a embedding con el mismo modelo que los documentos.
      FAISS devuelve los TOP_K fragmentos con menor distancia L2 (= más similares).

  A — Augmentation:
      Los fragmentos recuperados se insertan en el prompt junto con su metadata
      (fuente y página) para que el LLM pueda citar correctamente.

  G — Generation:
      El LLM recibe el sistema + contexto + pregunta y genera una respuesta
      fundamentada exclusivamente en los fragmentos recuperados.
      Esto reduce drásticamente las alucinaciones respecto a un LLM sin RAG.
"""


def rag_pdf(pregunta: str, k: int = TOP_K) -> dict:
    """
    Ejecuta el pipeline R-A-G completo para una pregunta.

    Parámetros:
      pregunta : texto libre de la consulta del usuario
      k        : número de fragmentos a recuperar (default: TOP_K)

    Retorna dict con:
      respuesta   : texto generado por el LLM
      fragmentos  : lista de fragmentos usados como contexto
    """
    # ── R: Retrieval ──────────────────────────────────────────────────────────
    query_vec      = np.array([get_embedding(pregunta)], dtype='float32')
    D, I           = index_pdf.search(query_vec, k=min(k, index_pdf.ntotal))
    top_fragmentos = [fragmentos[i] for i in I[0]]

    # ── A: Augmentation ───────────────────────────────────────────────────────
    # Cada fragmento va numerado con su fuente y página para facilitar citas.
    contexto = "\n\n".join(
        f"[{j+1}] Fuente: {f['fuente']}  |  Página: {f['pagina']}\n{f['texto']}"
        for j, f in enumerate(top_fragmentos)
    )
    prompt_usuario = f"Contexto recuperado del DOF:\n\n{contexto}\n\nPregunta: {pregunta}"

    # ── G: Generation ─────────────────────────────────────────────────────────
    respuesta = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt_usuario}
        ]
    )
    texto = respuesta.choices[0].message.content

    # --- Output ---
    print("\n" + "═" * 65)
    print(f"❓ Pregunta: {pregunta}")
    print(f"\n📄 Fragmentos recuperados (top-{k}):")
    for j, f in enumerate(top_fragmentos):
        preview = f['texto'][:110].replace('\n', ' ')
        print(f"  [{j+1}] {f['fuente']}  p.{f['pagina']}: {preview}...")
    print(f"\n🤖 Respuesta ({GENERATION_MODEL}):")
    print(f"   {texto}")
    print(f"\n📊 Tokens usados: {respuesta.usage.total_tokens}")

    return {"respuesta": texto, "fragmentos": top_fragmentos}


# --- Preguntas de ejemplo, una por cada artículo descargado ---

PREGUNTAS_DOF = [
    # SHCP — aguas nacionales
    "¿Qué establece la SHCP sobre las cuotas por trasvase de aguas nacionales?",
    # CONDUSEF — días inhábiles
    "¿Cuáles son los periodos de suspensión de términos y plazos de la CONDUSEF?",
    # Órgano de Administración Judicial
    "¿Qué cambios realiza el Órgano de Administración Judicial en la designación de titulares?",
    # Comisión Intersecretarial Mares y Costas
    "¿Qué secretarías integran la Comisión Intersecretarial para el Manejo Sustentable de Mares y Costas?",
    # NOM-004 expediente clínico
    "¿Qué información debe contener el expediente clínico según la NOM-004-SSA3-2012?",
]

print("\n" + "═" * 65)
print("📋 Preguntas de ejemplo sobre los artículos del DOF")
print("═" * 65)

for pregunta in PREGUNTAS_DOF:
    rag_pdf(pregunta)
    print()


# --- Modo interactivo ---

print("\n" + "═" * 65)
print("💬 Modo interactivo")
print(f"   Modelo embeddings : {EMBEDDING_MODEL}")
print(f"   Modelo generación : {GENERATION_MODEL}")
print(f"   Fragmentos en índice: {index_pdf.ntotal}  |  Top-K: {TOP_K}")
print("   Escribe tu pregunta o 'salir' para terminar.")
print("═" * 65)

while True:
    pregunta = input("\n❓ Tu pregunta: ").strip()
    if not pregunta:
        continue
    if pregunta.lower() in ("salir", "exit", "quit"):
        print("👋 Sesión terminada.")
        break
    rag_pdf(pregunta)

🔑 Introduce tu OpenAI API Key: ··········
📥 Descargando 5 artículos del DOF → /content/data

  → SHCP_aguas_nacionales_feb2026 (código 5780977)... ✓  (8.2 KB)
  → CONDUSEF_dias_inhabiles_ene2026 (código 5779201)... ✓  (7.3 KB)
  → DOF_OAJ_titulares_ene2026 (código 5778252)... ✓  (6.1 KB)
  → Comision_Mares_Costas_ago2022 (código 5660409)... ✓  (20.0 KB)
  → NOM004_expediente_clinico (código 5272787)... ✓  (49.9 KB)

✅ PDFs listos en /content/data/

📂 6 PDFs encontrados en /content/data:
  ✓ CONDUSEF_dias_inhabiles_ene2026.pdf: 3 páginas con texto
  ✓ Comision_Mares_Costas_ago2022.pdf: 19 páginas con texto
  ✓ DOF_14_enero_2026.pdf: 3 páginas con texto
  ✓ DOF_OAJ_titulares_ene2026.pdf: 3 páginas con texto
  ✓ NOM004_expediente_clinico.pdf: 27 páginas con texto
  ✓ SHCP_aguas_nacionales_feb2026.pdf: 5 páginas con texto

Total: 60 páginas extraídas

✂️  Chunking completado:
   Páginas     : 60
   Fragmentos  : 90
   Chunk size  : 500 tokens  |  Overlap: 50 tokens

🔢 Vectorizando 90 fragm